In [15]:
# =========================================================
# FULL SCRIPT BPS SCRAPING — AUTO-DISCOVERY TABLE ID
# =========================================================

import requests
import pandas as pd
import time
from tqdm import tqdm



In [20]:
import requests
import pandas as pd
import time
from tqdm import tqdm

# ================= 1. CONFIGURATION =====================
API_KEY = "d328d5f200379241367024848106698e"
YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
OUTPUT_FILE = "data_komoditas_bps_2020_2025.csv"

# --- Komoditas 1: Jumlah Penduduk ---
POPULATION_COMMODITY_NAME = "Jumlah Penduduk"
POPULATION_TABLE_IDS = [
    "WVRlTTcySlZDa3lUcFp6czNwbHl4QT09"
]
POPULATION_KEYWORDS = ["jumlah", "penduduk"]
POPULATION_COLUMN = "jumlah_penduduk"

# --- Komoditas 2: Populasi Ternak - Sapi Potong (Ekor) ---
CATTLE_POPULATION_NAME = "Populasi Ternak - Sapi Potong (Ekor)"
CATTLE_POPULATION_TABLE_IDS = [
    "S2ViU1dwVTlpSXRwU1MvendHN05Cdz09"
]
CATTLE_POPULATION_KEYWORDS = ["populasi", "sapi", "potong"]
CATTLE_POPULATION_COLUMN = "populasi_sapi_potong"

# --- Komoditas 3: Populasi Unggas - Ayam Pedaging (Ekor) ---
BROILER_POPULATION_NAME = "Populasi Unggas - Ayam Pedaging (Ekor)"
BROILER_POPULATION_TABLE_IDS = [
    "ckJyVXRMT05MWTNpaW9mdnVseFk0Zz09"
]
BROILER_POPULATION_KEYWORDS = ["populasi", "ayam", "pedaging"]
BROILER_POPULATION_COLUMN = "populasi_ayam_pedaging"

# --- Komoditas 4: Produksi Daging Sapi menurut Provinsi (Ton) ---
BEEF_PRODUCTION_NAME = "Produksi Daging Sapi menurut Provinsi (Ton)"
# This uses a different API endpoint, so no table ID, but a variable ID
BEEF_PRODUCTION_VAR_ID = 480 # From URL var/480
BEEF_PRODUCTION_KEYWORDS = ["produksi", "daging", "sapi"]
BEEF_PRODUCTION_COLUMN = "produksi_daging_sapi"

# --- Komoditas 5: Produksi Daging Unggas - Ayam Pedaging (kg) ---
BROILER_PRODUCTION_NAME = "Produksi Daging Unggas - Ayam Pedaging (kg)"
BROILER_PRODUCTION_TABLE_IDS = [
    "dWhmNFl6WXYyR093R2NjTGM3NG9idz09"
]
BROILER_PRODUCTION_KEYWORDS = ["produksi", "daging", "ayam", "pedaging"]
BROILER_PRODUCTION_COLUMN = "produksi_daging_ayam_pedaging"

In [27]:
# ================= 2. CORE FUNCTIONS ====================
def get_bps_data(year, table_id, kode_wilayah="0000000"):
    """Mengambil data BPS berdasarkan parameter dinamis (untuk SIMDASI)"""
    url = (
        f"https://webapi.bps.go.id/v1/api/interoperabilitas/datasource/simdasi/"
        f"id/25/tahun/{year}/id_tabel/{table_id}/"
        f"wilayah/{kode_wilayah}/key/{API_KEY}"
    )
    try:
        response = requests.get(url, timeout=30)
        return response.json()
    except Exception as e:
        # print(f"Error fetching SIMDASI data: {e}")
        return None

def get_bps_production_by_variable(year, var_id, kode_wilayah="0000000"):
    """Mengambil data BPS berdasarkan variable ID (untuk endpoint model/data)"""
    # For this endpoint, 'th' parameter expects an internal BPS year ID (e.g., 120 for 2020)
    bps_year_id = year - 1900 # Assuming this mapping, commonly found in BPS APIs
    url = (
        f"https://webapi.bps.go.id/v1/api/list/model/data/lang/ind/domain/{kode_wilayah}/"
        f"var/{var_id}/th/{bps_year_id}/key/{API_KEY}"
    )
    try:
        response = requests.get(url, timeout=30)
        return response.json()
    except Exception as e:
        # print(f"Error fetching Production data: {e}")
        return None

def get_target_var_id(main_data, target_keywords):
    """Mencari ID kode tersembunyi untuk variabel target berdasarkan kata kunci"""
    kolom_meta = main_data.get("kolom", {})
    for var_id, var_info in kolom_meta.items():
        nama_var = str(var_info.get("nama_variabel", "")).lower()
        # Check if all target keywords are present in the variable name
        if all(keyword in nama_var for keyword in target_keywords):
            return var_id
    return None

def parse_bps_simdasi_province_data(raw_data, year, nama_provinsi, target_keywords, value_column_name):
    """Parsing JSON untuk data SIMDASI, mencari data di level PROVINSI"""
    rows = []
    if not raw_data or raw_data.get("status") != "OK":
        return pd.DataFrame()

    try:
        main_data = raw_data.get("data", [{}, {}])[1]
        target_var_id = get_target_var_id(main_data, target_keywords)

        if not target_var_id:
            return pd.DataFrame()

        for item in main_data.get("data", []):
            nama_wilayah = item.get("label", "")
            kode_wilayah = item.get("kode_wilayah", "")

            # Filter hanya ambil baris yang merupakan Total Provinsi
            if nama_wilayah.lower() == nama_provinsi.lower():
                value = None
                if target_var_id in item.get("variables", {}):
                    value = item["variables"][target_var_id].get("value_raw", None)

                rows.append({
                    "tahun": year,
                    "provinsi": nama_provinsi,
                    "kode_wilayah": kode_wilayah,
                    value_column_name: value
                })
                break # Found the province total, no need to check other sub-regions
        return pd.DataFrame(rows)
    except Exception as e:
        # print(f"Error parsing SIMDASI data for {nama_provinsi}: {e}")
        return pd.DataFrame()

def parse_bps_production_province_data(raw_data, year, target_keywords, value_column_name):
    """Parsing JSON untuk data produksi, yang mungkin langsung per provinsi"""
    rows = []
    if not raw_data or raw_data.get("status") != "OK":
        return pd.DataFrame()

    try:
        main_data = raw_data.get("data", [{}, {}])[1]
        target_var_id = get_target_var_id(main_data, target_keywords)

        if not target_var_id:
            return pd.DataFrame()

        for item in main_data.get("data", []):
            nama_wilayah = item.get("label", "")
            kode_wilayah = item.get("kode_wilayah", "")

            # Assuming data is directly at province level in 'data' list
            # The BPS production API often returns a list of provinces directly
            if kode_wilayah != "0000000" and "indonesia" not in nama_wilayah.lower(): # Exclude national total if present
                value = None
                if target_var_id in item.get("variables", {}):
                    value = item["variables"][target_var_id].get("value_raw", None)

                if value is not None: # Only add if a value exists
                    rows.append({
                        "tahun": year,
                        "provinsi": nama_wilayah,
                        "kode_wilayah": kode_wilayah,
                        value_column_name: value
                    })
        return pd.DataFrame(rows)
    except Exception as e:
        # print(f"Error parsing Production data: {e}")
        return pd.DataFrame()

In [ ]:
# ================= 3. MAIN LOOP & AUTO-DISCOVERY ========
all_df_raw_commodities = [] # This will temporarily store dfs for each commodity
print("🚀 Memulai proses scraping untuk semua komoditas...")

COMMODITIES_TO_SCRAPE = [
    {
        "name": POPULATION_COMMODITY_NAME,
        "type": "simdasi",
        "table_ids": POPULATION_TABLE_IDS,
        "keywords": POPULATION_KEYWORDS,
        "column_name": POPULATION_COLUMN
    },
    {
        "name": CATTLE_POPULATION_NAME,
        "type": "simdasi",
        "table_ids": CATTLE_POPULATION_TABLE_IDS,
        "keywords": CATTLE_POPULATION_KEYWORDS,
        "column_name": CATTLE_POPULATION_COLUMN
    },
    {
        "name": BROILER_POPULATION_NAME,
        "type": "simdasi",
        "table_ids": BROILER_POPULATION_TABLE_IDS,
        "keywords": BROILER_POPULATION_KEYWORDS,
        "column_name": BROILER_POPULATION_COLUMN
    },
    {
        "name": BEEF_PRODUCTION_NAME,
        "type": "production_var",
        "var_id": BEEF_PRODUCTION_VAR_ID,
        "keywords": BEEF_PRODUCTION_KEYWORDS,
        "column_name": BEEF_PRODUCTION_COLUMN
    },
    {
        "name": BROILER_PRODUCTION_NAME,
        "type": "simdasi",
        "table_ids": BROILER_PRODUCTION_TABLE_IDS,
        "keywords": BROILER_PRODUCTION_KEYWORDS,
        "column_name": BROILER_PRODUCTION_COLUMN
    }
]

# Create an empty DataFrame to store all results
final_merged_df = pd.DataFrame()

for commodity_info in COMMODITIES_TO_SCRAPE:
    commodity_name = commodity_info["name"]
    commodity_type = commodity_info["type"]
    commodity_keywords = commodity_info["keywords"]
    commodity_column_name = commodity_info["column_name"]

    print(f"\n--- Mengambil data untuk: {commodity_name} ---")

    commodity_dfs_for_all_years = [] # To store dataframes for the current commodity across years

    for year in YEARS:
        print(f"[{year}] Mencari data untuk {commodity_name}...")

        raw_data_for_year = None
        active_id = None # Could be table_id or var_id

        if commodity_type == "simdasi":
            # For SIMDASI, iterate through table IDs to find an active one
            for tbl_id in commodity_info["table_ids"]:
                temp_data = get_bps_data(year, tbl_id, "0000000") # Get national data to check validity
                if temp_data and temp_data.get("status") == "OK":
                    main_data_check = temp_data.get("data", [{}, {}])[1]
                    # Check if the target variable exists in this table
                    if get_target_var_id(main_data_check, commodity_keywords):
                        active_id = tbl_id
                        raw_data_for_year = temp_data # Store national raw data
                        break
            if not active_id:
                print(f"  ❌ Gagal: Tidak ada tabel yang mengandung data '{commodity_name}' di tahun {year}.\n")
                continue
            print(f"  ✅ Tabel valid ditemukan ({active_id}). Mulai drill-down wilayah...")

            # Extract list of provinces from national data (assuming SIMDASI structure)
            main_data_nasional = raw_data_for_year.get("data", [{}, {}])[1]
            list_provinsi = [
                {"nama": item.get("label", ""), "kode": item.get("kode_wilayah", "")}
                for item in main_data_nasional.get("data", [])
                if item.get("label", "").lower() != "indonesia" and item.get("kode_wilayah", "")
            ]

            # Loop to get data per province
            df_year_list_per_commodity_per_year = []
            for prov in tqdm(list_provinsi, desc=f"Scraping {year} {commodity_name}"):
                time.sleep(0.5) # Be kind to the API
                raw_prov = get_bps_data(year, active_id, prov["kode"])
                df_prov = parse_bps_simdasi_province_data(raw_prov, year, prov["nama"], commodity_keywords, commodity_column_name)
                if not df_prov.empty:
                    df_year_list_per_commodity_per_year.append(df_prov)
            if df_year_list_per_commodity_per_year:
                commodity_dfs_for_all_years.append(pd.concat(df_year_list_per_commodity_per_year, ignore_index=True))

        elif commodity_type == "production_var":
            # For production_var, directly use the var_id
            active_id = commodity_info["var_id"]
            raw_data_for_year = get_bps_production_by_variable(year, active_id, "0000000") # National domain or specific domain if needed

            if not raw_data_for_year or raw_data_for_year.get("status") != "OK":
                print(f"  ❌ Gagal: Tidak ada data '{commodity_name}' untuk variable ID {active_id} di tahun {year}.\n")
                continue
            print(f"  ✅ Data ditemukan untuk variable ID {active_id}. Mulai parsing wilayah...")

            # The production API often returns provincial data directly, so parse it
            df_prov_data = parse_bps_production_province_data(raw_data_for_year, year, commodity_keywords, commodity_column_name)
            if not df_prov_data.empty:
                commodity_dfs_for_all_years.append(df_prov_data)

        print("") # Spasi antar tahun

    if commodity_dfs_for_all_years:
        # Concatenate all years for the current commodity
        commodity_full_df = pd.concat(commodity_dfs_for_all_years, ignore_index=True)

        if final_merged_df.empty:
            final_merged_df = commodity_full_df
        else:
            # Merge with existing dataframe based on 'tahun', 'provinsi', 'kode_wilayah'
            final_merged_df = pd.merge(final_merged_df, commodity_full_df,
                                     on=['tahun', 'provinsi', 'kode_wilayah'],
                                     how='outer')

print("\n🏁 Proses scraping selesai untuk semua komoditas.")
all_df = [final_merged_df] # Assign the final merged df to all_df for the next cell

🚀 Memulai proses scraping untuk semua komoditas...

--- Mengambil data untuk: Jumlah Penduduk ---
[2020] Mencari data untuk Jumlah Penduduk...
  ✅ Tabel valid ditemukan (WVRlTTcySlZDa3lUcFp6czNwbHl4QT09). Mulai drill-down wilayah...


Scraping 2020 Jumlah Penduduk: 100%|██████████| 34/34 [01:05<00:00,  1.92s/it]



[2021] Mencari data untuk Jumlah Penduduk...
  ✅ Tabel valid ditemukan (WVRlTTcySlZDa3lUcFp6czNwbHl4QT09). Mulai drill-down wilayah...


Scraping 2021 Jumlah Penduduk: 100%|██████████| 34/34 [01:09<00:00,  2.04s/it]



[2022] Mencari data untuk Jumlah Penduduk...
  ✅ Tabel valid ditemukan (WVRlTTcySlZDa3lUcFp6czNwbHl4QT09). Mulai drill-down wilayah...


...

Scraping 2024 Jumlah Penduduk: 100%|██████████| 38/38 [01:27<00:00,  2.30s/it]



[2025] Mencari data untuk Jumlah Penduduk...
  ✅ Tabel valid ditemukan (WVRlTTcySlZDa3lUcFp6czNwbHl4QT09). Mulai drill-down wilayah...


Scraping 2025 Jumlah Penduduk: 100%|██████████| 38/38 [01:12<00:00,  1.90s/it]




--- Mengambil data untuk: Populasi Ternak - Sapi Potong (Ekor) ---
[2020] Mencari data untuk Populasi Ternak - Sapi Potong (Ekor)...
  ✅ Tabel valid ditemukan (S2ViU1dwVTlpSXRwU1MvendHN05Cdz09). Mulai drill-down wilayah...


Scraping 2020 Populasi Ternak - Sapi Potong (Ekor):  56%|█████▌    | 19/34 [00:35<00:29,  1.99s/it]

In [29]:
# ================= 4. CLEANING & EXPORT =================
if all_df:
    print("🧹 Membersihkan dan merapikan data provinsi...")
    final_df = all_df[0] # all_df now contains a single merged DataFrame

    # List of all new commodity columns
    commodity_value_columns = [
        POPULATION_COLUMN,
        CATTLE_POPULATION_COLUMN,
        BROILER_POPULATION_COLUMN,
        BEEF_PRODUCTION_COLUMN,
        BROILER_PRODUCTION_COLUMN
    ]

    # Apply cleaning for each commodity column
    for col in commodity_value_columns:
        if col in final_df.columns:
            final_df[col] = final_df[col].astype(str)
            final_df[col] = (
                final_df[col]
                .str.replace('.', '', regex=False) # Remove thousands separator
                .str.replace(',', '.', regex=False) # Replace comma decimal with dot decimal
            )
            final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

    # Drop rows where all commodity values are NaN (if any row has no data for any commodity)
    existing_commodity_columns = [col for col in commodity_value_columns if col in final_df.columns]
    if existing_commodity_columns: # Only call dropna if there are columns to consider
        final_df = final_df.dropna(subset=existing_commodity_columns, how='all')
    final_df = final_df.drop_duplicates().reset_index(drop=True)

    final_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"✨ SELESAI! Total {len(final_df)} data provinsi berhasil disimpan.")
    display(final_df.sort_values(['tahun', 'provinsi']))
else:
    print("❌ Tidak ada data. Pastikan Step 3 sudah dijalankan.")

🧹 Membersihkan dan merapikan data provinsi...
✨ SELESAI! Total 216 data provinsi berhasil disimpan.


,tahun,provinsi,kode_wilayah,jumlah_penduduk,populasi_sapi_potong,populasi_ayam_pedaging,produksi_daging_ayam_pedaging
0,2020,Aceh,1100000,5274.9,435376.0,32590982.0,3.593547e+07
1,2020,Bali,5100000,4317.4,550350.0,71729771.0,7.909068e+07
2,2020,Banten,3600000,11904.6,41899.0,196970599.0,2.171837e+08
3,2020,Bengkulu,1700000,2010.7,154405.0,8663406.0,9.552445e+06
4,2020,DI Yogyakarta,3400000,3668719.0,309259.0,51674388.0,5.697721e+07
...,...,...,...,...,...,...,...
211,2025,Sulawesi Tenggara,7400000,2836.7,275696.0,8671522.0,1.059781e+07
212,2025,Sulawesi Utara,7100000,2721.4,63985.0,11879472.0,1.451837e+07
213,2025,Sumatera Barat,1300000,5914.3,411046.0,55997358.0,6.843658e+07
214,2025,Sumatera Selatan,1600000,8928.5,283947.0,113038192.0,1.381484e+08
